# 🏀 NBA Player Efficiency & Value Dashboard
### 2025–26 Season Analysis

This notebook walks through a full data analysis pipeline:
1. **Load & clean** the player stats CSV
2. **Engineer a Value Score** metric (per-36-minute formula)
3. **Tier players** into Star / Starter / Rotation / Bench
4. **Visualise** four charts into a polished dashboard
5. **Export** a top-50 ranked CSV

**Libraries used:** `pandas`, `matplotlib`, `seaborn`

---
## Cell 1 — Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries loaded")

---
## Cell 2 — Load & Clean the Data

We load the CSV and filter out players with very few minutes played.  
**400 minutes** ≈ roughly 6+ minutes per game across a full season — below that, stats become noisy and unreliable.

In [ ]:
# ── Load the data ──────────────────────────────────────────────────────────────
# Update this path to wherever your CSV lives
df = pd.read_csv("nba_player_stats_2026.csv")

print(f"Raw dataset: {len(df)} players, {df.shape[1]} columns")
df.head(3)

In [ ]:
# ── Filter low-minute players ──────────────────────────────────────────────────
df = df[df["MIN"] >= 400].copy()
df = df.reset_index(drop=True)

print(f"After filtering (400+ minutes): {len(df)} players remain")

# Quick check for missing values
print("\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## Cell 3 — Calculate Value Score (Per 36 Minutes)

**Why per-36?**  
A player scoring 20 points in 20 minutes is more efficient than one scoring 20 points in 35 minutes. Per-36-minute stats normalise everyone to the same playing time so comparisons are fair.

**Value Score formula:**
```
Value = (PTS×1.0) + (REB×0.8) + (AST×1.2) + (STL×1.5) + (BLK×1.5) − (TOV×1.2)
```
Weights reflect real basketball value — playmaking and defence are slightly rewarded, turnovers penalised.

In [ ]:
# ── Minutes per game ───────────────────────────────────────────────────────────
df["MPG"] = (df["MIN"] / df["GP"]).round(1)

# ── Per-36 helper ──────────────────────────────────────────────────────────────
# This lambda function converts any season total into a per-36-minute rate
per36 = lambda col: (df[col] / df["MIN"]) * 36

df["PTS_36"] = per36("PTS")
df["REB_36"] = per36("REB")
df["AST_36"] = per36("AST")
df["STL_36"] = per36("STL")
df["BLK_36"] = per36("BLK")
df["TOV_36"] = per36("TOV")

# ── Value Score ────────────────────────────────────────────────────────────────
df["VALUE_SCORE"] = (
    df["PTS_36"]  * 1.0 +
    df["REB_36"]  * 0.8 +
    df["AST_36"]  * 1.2 +
    df["STL_36"]  * 1.5 +
    df["BLK_36"]  * 1.5 -
    df["TOV_36"]  * 1.2
).round(2)

print("Value Score — top 5 players:")
df[["PLAYER", "TEAM", "MPG", "VALUE_SCORE"]].nlargest(5, "VALUE_SCORE")

---
## Cell 4 — Assign Player Tiers

We bucket each player into one of four tiers based on their Value Score:

| Tier | Value Score |
|------|-------------|
| ⭐ Star | ≥ 35 |
| 🔵 Starter | 28 – 34.9 |
| 🟡 Rotation | 22 – 27.9 |
| ⚪ Bench | < 22 |

In [ ]:
def assign_tier(score):
    if score >= 35:
        return "⭐ Star"
    elif score >= 28:
        return "🔵 Starter"
    elif score >= 22:
        return "🟡 Rotation"
    else:
        return "⚪ Bench"

df["TIER"] = df["VALUE_SCORE"].apply(assign_tier)

print("Player count per tier:")
print(df["TIER"].value_counts())

---
## Cell 5 — Visual Style Setup

We define colours, fonts and a dark theme before building any charts.  
Keeping these in one place makes it easy to restyle the whole dashboard later.

In [ ]:
# ── Colour palette — one colour per tier ──────────────────────────────────────
TIER_COLORS = {
    "⭐ Star":      "#F5A623",   # gold
    "🔵 Starter":  "#4A90D9",   # blue
    "🟡 Rotation": "#7ED321",   # green
    "⚪ Bench":     "#9B9B9B",   # grey
}

TIER_ORDER = ["⭐ Star", "🔵 Starter", "🟡 Rotation", "⚪ Bench"]

# ── Dark background theme ─────────────────────────────────────────────────────
plt.style.use("dark_background")
BG_COLOR   = "#0F1117"
CARD_COLOR = "#1A1D2E"
TEXT_COLOR = "#E8E8F0"

plt.rcParams.update({
    "font.family":     "DejaVu Sans",
    "text.color":      TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "xtick.color":     TEXT_COLOR,
    "ytick.color":     TEXT_COLOR,
})

# ── Reusable helper to style each chart panel ─────────────────────────────────
def style_ax(ax, title):
    ax.set_facecolor(CARD_COLOR)
    ax.set_title(title, fontsize=13, fontweight="bold", color=TEXT_COLOR, pad=12)
    ax.spines[["top","right","left","bottom"]].set_color("#2A2D3E")
    ax.tick_params(colors=TEXT_COLOR, labelsize=9)

print("✅ Style configured")

---
## Cell 6 — Chart 1: Top 20 Players by Value Score

A horizontal bar chart of the 20 highest-scoring players.  
Bars are coloured by tier so you can instantly see the breakdown.

In [ ]:
from matplotlib.patches import Patch

top20 = df.nlargest(20, "VALUE_SCORE").sort_values("VALUE_SCORE")
bar_colors = [TIER_COLORS[t] for t in top20["TIER"]]

fig, ax = plt.subplots(figsize=(14, 7), facecolor=BG_COLOR)

bars = ax.barh(top20["PLAYER"], top20["VALUE_SCORE"],
               color=bar_colors, edgecolor="none", height=0.7)

# Value labels at the end of each bar
for bar, score in zip(bars, top20["VALUE_SCORE"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{score:.1f}", va="center", ha="left",
            fontsize=9, color=TEXT_COLOR, fontweight="bold")

ax.set_xlim(0, top20["VALUE_SCORE"].max() + 5)
ax.set_xlabel("Value Score (per 36 min)", fontsize=11)
style_ax(ax, "🏆  Top 20 Players by Value Score")

legend_handles = [Patch(color=TIER_COLORS[t], label=t) for t in TIER_ORDER]
ax.legend(handles=legend_handles, loc="lower right",
          framealpha=0.2, fontsize=9, labelcolor=TEXT_COLOR)

plt.tight_layout()
plt.show()

---
## Cell 7 — Chart 2: Efficiency vs. Scoring (Scatter Plot)

Each dot is a player. The x-axis shows total points scored; the y-axis shows the NBA's built-in Efficiency Rating (EFF).  
The top 5 players by Value Score are labelled — notice they cluster in the top-right.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), facecolor=BG_COLOR)

for tier in TIER_ORDER:
    mask = df["TIER"] == tier
    ax.scatter(
        df.loc[mask, "PTS"],
        df.loc[mask, "EFF"],
        c=TIER_COLORS[tier],
        label=tier,
        alpha=0.75,
        s=45,
        edgecolors="none"
    )

# Label the top 5 players
top5 = df.nlargest(5, "VALUE_SCORE")
for _, row in top5.iterrows():
    ax.annotate(row["PLAYER"].split()[-1],
                xy=(row["PTS"], row["EFF"]),
                xytext=(5, 4), textcoords="offset points",
                fontsize=8, color=TEXT_COLOR, alpha=0.9)

ax.set_xlabel("Total Points (season)", fontsize=11)
ax.set_ylabel("NBA Efficiency Rating", fontsize=11)
ax.legend(fontsize=9, labelcolor=TEXT_COLOR, framealpha=0.2)
style_ax(ax, "📊  Efficiency vs. Scoring")

plt.tight_layout()
plt.show()

---
## Cell 8 — Chart 3: Team Roster Depth by Tier

A stacked bar chart showing how each team's roster breaks down across tiers.  
Teams are sorted by Star + Starter count — the most "loaded" rosters appear at the top.

In [ ]:
# Count players per tier per team
team_tiers = (
    df.groupby(["TEAM", "TIER"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=TIER_ORDER, fill_value=0)
)

# Sort by star power
team_tiers["_sort"] = team_tiers["⭐ Star"] * 3 + team_tiers["🔵 Starter"]
team_tiers = team_tiers.sort_values("_sort", ascending=True).drop(columns="_sort")

fig, ax = plt.subplots(figsize=(10, 9), facecolor=BG_COLOR)

team_tiers.plot(
    kind="barh",
    stacked=True,
    color=[TIER_COLORS[t] for t in TIER_ORDER],
    ax=ax,
    legend=True,
    width=0.75
)

ax.set_xlabel("Number of Players", fontsize=11)
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=8)
ax.legend(fontsize=9, labelcolor=TEXT_COLOR, framealpha=0.2)
style_ax(ax, "🏀  Team Roster Depth by Tier")

plt.tight_layout()
plt.show()

---
## Cell 9 — Chart 4: Hidden Gems

Players getting **15–28 minutes per game** (limited role) but ranking highly on Value Score.  
These are the underutilised players — potential breakout candidates or trade targets.

In [ ]:
gems_pool = df[(df["MPG"] >= 15) & (df["MPG"] <= 28)].copy()
top_gems = gems_pool.nlargest(18, "VALUE_SCORE").sort_values("VALUE_SCORE")
gem_colors = [TIER_COLORS[t] for t in top_gems["TIER"]]

fig, ax = plt.subplots(figsize=(14, 7), facecolor=BG_COLOR)

bars = ax.barh(
    top_gems["PLAYER"] + "  (" + top_gems["TEAM"] + ")",
    top_gems["VALUE_SCORE"],
    color=gem_colors, edgecolor="none", height=0.7
)

# Show minutes per game inside each bar
for bar, (_, row) in zip(bars, top_gems.iterrows()):
    ax.text(0.4, bar.get_y() + bar.get_height() / 2,
            f"{row['MPG']:.0f} mpg",
            va="center", ha="left",
            fontsize=8, color="white", alpha=0.85)

ax.set_xlabel("Value Score (per 36 min)", fontsize=11)
style_ax(ax, "💎  Hidden Gems — High Value, Limited Minutes (15–28 mpg)")

plt.tight_layout()
plt.show()

---
## Cell 10 — Full Dashboard (All 4 Charts Together)

This cell combines all four charts into a single publication-ready figure — ideal for exporting as a PNG for your portfolio.

In [ ]:
fig = plt.figure(figsize=(20, 18), facecolor=BG_COLOR)
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :])   # full-width top
ax2 = fig.add_subplot(gs[1, 0])   # middle left
ax3 = fig.add_subplot(gs[1, 1])   # middle right
ax4 = fig.add_subplot(gs[2, :])   # full-width bottom

# ── Chart 1: Top 20 ────────────────────────────────────────────────────────────
top20 = df.nlargest(20, "VALUE_SCORE").sort_values("VALUE_SCORE")
bars = ax1.barh(top20["PLAYER"], top20["VALUE_SCORE"],
                color=[TIER_COLORS[t] for t in top20["TIER"]],
                edgecolor="none", height=0.7)
for bar, score in zip(bars, top20["VALUE_SCORE"]):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
             f"{score:.1f}", va="center", ha="left", fontsize=8.5,
             color=TEXT_COLOR, fontweight="bold")
ax1.set_xlim(0, top20["VALUE_SCORE"].max() + 5)
ax1.set_xlabel("Value Score (per 36 min)", fontsize=10)
style_ax(ax1, "🏆  Top 20 Players by Value Score")
legend_handles = [Patch(color=TIER_COLORS[t], label=t) for t in TIER_ORDER]
ax1.legend(handles=legend_handles, loc="lower right", framealpha=0.2,
           fontsize=9, labelcolor=TEXT_COLOR)

# ── Chart 2: Efficiency vs Points ──────────────────────────────────────────────
for tier in TIER_ORDER:
    mask = df["TIER"] == tier
    ax2.scatter(df.loc[mask, "PTS"], df.loc[mask, "EFF"],
                c=TIER_COLORS[tier], label=tier, alpha=0.75, s=40, edgecolors="none")
top5 = df.nlargest(5, "VALUE_SCORE")
for _, row in top5.iterrows():
    ax2.annotate(row["PLAYER"].split()[-1],
                 xy=(row["PTS"], row["EFF"]),
                 xytext=(5, 4), textcoords="offset points",
                 fontsize=7.5, color=TEXT_COLOR, alpha=0.9)
ax2.set_xlabel("Total Points (season)", fontsize=10)
ax2.set_ylabel("NBA Efficiency Rating", fontsize=10)
ax2.legend(fontsize=8, labelcolor=TEXT_COLOR, framealpha=0.2)
style_ax(ax2, "📊  Efficiency vs. Scoring")

# ── Chart 3: Team depth ─────────────────────────────────────────────────────────
team_tiers = (
    df.groupby(["TEAM", "TIER"]).size()
    .unstack(fill_value=0)
    .reindex(columns=TIER_ORDER, fill_value=0)
)
team_tiers["_sort"] = team_tiers["⭐ Star"] * 3 + team_tiers["🔵 Starter"]
team_tiers = team_tiers.sort_values("_sort", ascending=True).drop(columns="_sort")
team_tiers.plot(kind="barh", stacked=True,
                color=[TIER_COLORS[t] for t in TIER_ORDER],
                ax=ax3, legend=False, width=0.75)
ax3.set_xlabel("Number of Players", fontsize=10)
ax3.set_ylabel("")
ax3.tick_params(axis="y", labelsize=7.5)
style_ax(ax3, "🏀  Team Roster Depth by Tier")

# ── Chart 4: Hidden Gems ────────────────────────────────────────────────────────
gems_pool = df[(df["MPG"] >= 15) & (df["MPG"] <= 28)].copy()
top_gems  = gems_pool.nlargest(18, "VALUE_SCORE").sort_values("VALUE_SCORE")
bars4 = ax4.barh(
    top_gems["PLAYER"] + "  (" + top_gems["TEAM"] + ")",
    top_gems["VALUE_SCORE"],
    color=[TIER_COLORS[t] for t in top_gems["TIER"]],
    edgecolor="none", height=0.7
)
for bar, (_, row) in zip(bars4, top_gems.iterrows()):
    ax4.text(0.4, bar.get_y() + bar.get_height() / 2,
             f"{row['MPG']:.0f} mpg", va="center", ha="left",
             fontsize=7.5, color="white", alpha=0.8)
ax4.set_xlabel("Value Score (per 36 min)", fontsize=10)
style_ax(ax4, "💎  Hidden Gems — High Value, Limited Minutes (15–28 mpg)")

# ── Title & save ───────────────────────────────────────────────────────────────
fig.suptitle("NBA Player Efficiency & Value Dashboard  ·  2025–26 Season",
             fontsize=18, fontweight="bold", color=TEXT_COLOR, y=0.98)
fig.text(0.5, 0.965,
         "Value Score = weighted per-36 formula  |  filtered to players with 400+ total minutes",
         ha="center", fontsize=9, color="#888899")

plt.savefig("nba_dashboard.png", dpi=150, bbox_inches="tight",
            facecolor=BG_COLOR, edgecolor="none")
print("✅ Dashboard saved as nba_dashboard.png")
plt.show()

---
## Cell 11 — Export Top 50 Players to CSV

In [ ]:
top50_cols = ["RANK","PLAYER","TEAM","GP","MPG","PTS","REB","AST",
              "STL","BLK","TOV","EFF","VALUE_SCORE","TIER"]

top50 = (
    df[top50_cols]
    .nlargest(50, "VALUE_SCORE")
    .reset_index(drop=True)
)

top50.to_csv("nba_top50_value.csv", index=False)
print("✅ Saved: nba_top50_value.csv")
top50.head(10)